# Exploração dos dados: carga, temperatura e PLD

Primeira análise exploratória da Sprint 2 (tarefa 2.8). **Lê só das tabelas do dataset `marts`**
(dimensões e fatos do dbt), com o teto de custo de `ingestion.common.gcp.executar_consulta`
(200 MiB por consulta; o BigQuery recusa a que passar disso). Uma consulta que leia raw ou staging
é recusada antes de ser enviada.

Cada seção tem: o que o gráfico mostra (fonte, filtro e método), a consulta e o gráfico, e uma célula
**Minhas conclusões** para escrever o que os dados dizem. As figuras são salvas em `docs/figuras/`
para o README.

Como rodar: `uv run --env-file .env --with jupyterlab jupyter lab notebooks/` (precisa do
`gcloud auth application-default login`) ou escolha o kernel `.venv` no VS Code. O arquivo no git
fica **sem saídas** (`nbstripout`); as imagens estão em `docs/figuras/`.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from notebooks import utilitarios as u

u.aplicar_estilo()
sessao = u.Sessao()  # lê o .env (projeto) e usa o login do gcloud; sem arquivo de chave
T = sessao.tabela  # T('fct_carga_horaria') -> `projeto.marts.fct_carga_horaria`

## 1. Carga média mensal do Sudeste/Centro-Oeste, 2000 a 2026

**Fonte:** `fct_carga_horaria` (ONS, subsistema `SE`) com `dim_tempo` para o mês local.
**Método:** média das cargas horárias de cada mês (MWmed); a linha grossa é a média móvel de 12
meses. As faixas e linhas marcam eventos; as datas com `~` são **aproximadas** (ver `EVENTOS` no
código), e a da MMGD vem da documentação do ONS. O último mês é parcial (marcador vazado).

In [ ]:
carga_mensal = sessao.consultar(
    "1. carga mensal do SE",
    f"""
    SELECT DATE_TRUNC(t.data_local, MONTH) AS mes,
           AVG(c.carga_mwmed) AS carga_media,
           COUNT(c.carga_mwmed) AS horas
    FROM {T("fct_carga_horaria")} AS c
    JOIN {T("dim_tempo")} AS t USING (instante_utc)
    WHERE c.codigo_submercado = 'SE'
    GROUP BY mes
    ORDER BY mes
    """,
)
carga_mensal["mes"] = pd.to_datetime(carga_mensal["mes"])
carga_mensal = carga_mensal.dropna(subset=["carga_media"]).reset_index(
    drop=True
)  # dez/1999 só tem 2 h sem carga
carga_mensal["carga_media"] = carga_mensal["carga_media"].astype(float)
carga_mensal["media_12m"] = carga_mensal["carga_media"].rolling(12).mean()

# (rótulo, início, fim ou None para uma linha, data aproximada?)
EVENTOS = [
    ("Racionamento", "2001-06-01", "2002-02-28", True),  # ~ jun/2001 a fev/2002
    (
        "Crise de 2009",
        "2008-10-01",
        "2009-06-30",
        True,
    ),  # ~ faixa que cobre a queda a partir de dez/2008
    ("Pandemia", "2020-03-01", None, True),  # ~ início em mar/2020
    ("MMGD entra na carga", "2023-04-29", None, False),  # data da documentação do ONS
]

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(
    carga_mensal["mes"],
    carga_mensal["carga_media"],
    color=u.AZUL,
    alpha=0.35,
    linewidth=1.0,
    label="média do mês",
)
ax.plot(
    carga_mensal["mes"],
    carga_mensal["media_12m"],
    color=u.AZUL,
    linewidth=2.2,
    label="média móvel de 12 meses",
)
ultimo = carga_mensal.iloc[-1]
if ultimo["horas"] < 0.9 * ultimo["mes"].days_in_month * 24:  # mês corrente incompleto
    ax.plot(
        ultimo["mes"],
        ultimo["carga_media"],
        "o",
        mfc=u.SUPERFICIE,
        mec=u.AZUL,
        mew=1.6,
        ms=6,
        label="mês parcial",
    )
ymax = carga_mensal["carga_media"].max()
ax.set_ylim(carga_mensal["carga_media"].min() * 0.9, ymax * 1.16)
for i, (rotulo, ini, fim, aprox) in enumerate(EVENTOS):
    x0 = pd.Timestamp(ini)
    if fim:
        ax.axvspan(x0, pd.Timestamp(fim), color=u.TINTA_3, alpha=0.15, linewidth=0)
    else:
        ax.axvline(x0, color=u.TINTA_3, linewidth=1.0, linestyle=(0, (4, 3)))
    ax.annotate(
        ("~ " if aprox else "") + rotulo,
        (x0, ymax * (1.13 - 0.045 * (i % 2))),
        fontsize=8.5,
        color=u.TINTA_2,
        ha="left",
        va="top",
        xytext=(3, 0),
        textcoords="offset points",
    )
ax.set_xlim(pd.Timestamp("2000-01-01"), pd.Timestamp("2026-12-31"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("carga média (MWmed)")
ax.set_title("Carga do Sudeste/Centro-Oeste: média mensal, 2000 a 2026")
ax.legend(loc="lower right", fontsize=9, ncol=3)
fig.text(
    0.01,
    -0.02,
    "~ data aproximada. Fonte: ONS (curva de carga horária), via marts.fct_carga_horaria.",
    fontsize=8,
    color=u.TINTA_3,
)
u.salvar_figura(fig, "01_carga_mensal_se")
plt.show()

### Minhas conclusões

A carga do SE/CO cresceu de ~26 mil MWmed em 2000 para ~44 mil em 2025, mas não de forma contínua. Os choques são visíveis: o racionamento de 2001 derrubou a carga em ~26% em poucos meses (mínima mensal de jul/2001 contra a média de jan–mai/2001), e a média móvel só superou o pico anterior em mai/2004; a crise de 2008/2009 e a pandemia (abr/2020 −14,5% e mai/2020 −12,6% contra o mesmo mês de 2019) foram quedas mais curtas, recuperadas em menos de um ano. Entre 2014 e 2019 a carga ficou praticamente parada (+2,4% em cinco anos; recessão).

Dois pontos importam para o projeto:

1. A sazonalidade ficou mais forte com o tempo. Medida como o **pico contra o vale mensal do mesmo ano** (maior média mensal ÷ menor média mensal − 1), ela era de 8,8% em 2005 e chegou a 25,5% em 2019, o que é coerente com mais uso de ar-condicionado.
2. A carga de 2023–2024 coincide em parte com a inclusão da MMGD a partir de 29/04/2023, mas o efeito não se separa de clima e economia só com esta série: a variação contra o mesmo mês do ano anterior foi só +4,0% em mai/2023, +3,1% em jun e +1,4% em jul, e só passou de +12% a partir de set/2023.

O modelo de previsão mensal precisa tratar o possível degrau (variável de mudança de nível ou ajuste da série), além dos choques de 2001 e 2020 como períodos atípicos.

## 2. Perfil médio da carga por hora do dia e tipo de dia

**Fonte:** `fct_carga_horaria` (SE) com `dim_tempo` (`hora_local`, `tipo_dia`).
**Método:** média da carga em cada hora **local** do dia, separada em dia útil, sábado e
domingo/feriado (feriado nacional da biblioteca `holidays`, que cai como `domingo_feriado` mesmo
num sábado). Período **2021 a 2025** (a janela do backtest; sem horário de verão, então a hora
local é sempre UTC−3). Mude `ANO_INICIO` e `ANO_FIM` para outro período.

In [ ]:
ANO_INICIO, ANO_FIM = 2021, 2025  # anos inteiros, inclusive

perfil = sessao.consultar(
    "2. perfil por hora do dia",
    f"""
    SELECT t.tipo_dia, t.hora_local, AVG(c.carga_mwmed) AS carga_media
    FROM {T("fct_carga_horaria")} AS c
    JOIN {T("dim_tempo")} AS t USING (instante_utc)
    WHERE c.codigo_submercado = 'SE'
      AND c.instante_utc >= TIMESTAMP('{ANO_INICIO}-01-01 00:00:00', 'America/Sao_Paulo')
      AND c.instante_utc <  TIMESTAMP('{ANO_FIM + 1}-01-01 00:00:00', 'America/Sao_Paulo')
    GROUP BY t.tipo_dia, t.hora_local
    ORDER BY t.tipo_dia, t.hora_local
    """,
)
perfil["carga_media"] = perfil["carga_media"].astype(float)

TIPOS = [
    ("dia_util", "Dia útil", u.AZUL),
    ("sabado", "Sábado", u.LARANJA),
    ("domingo_feriado", "Domingo e feriado", u.VERDE_AGUA),
]
fig, ax = plt.subplots(figsize=(9, 4.8))
for chave, rotulo, cor in TIPOS:
    d = perfil[perfil["tipo_dia"] == chave]
    ax.plot(d["hora_local"], d["carga_media"], color=cor, label=rotulo)
    ax.annotate(
        rotulo,
        (d["hora_local"].iloc[-1], d["carga_media"].iloc[-1]),
        xytext=(6, 0),
        textcoords="offset points",
        fontsize=9,
        color=u.TINTA_2,
        va="center",
    )  # rótulo direto (relief do contraste)
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("hora local do dia (início da hora)")
ax.set_ylabel("carga média (MWmed)")
ax.set_title(f"Perfil da carga do SE/CO por hora do dia, {ANO_INICIO} a {ANO_FIM}")
ax.legend(loc="lower right", fontsize=9)
fig.subplots_adjust(right=0.82)
u.salvar_figura(fig, "02_perfil_horario_por_tipo_de_dia")
plt.show()

### Minhas conclusões

Em dia útil, a carga vai de ~35,6 mil MWmed às 3h para ~49,2 mil às 19h, uma variação de ~38% dentro do dia. Há duas quedas pequenas (~1%), às 12h (almoço) e às 17h (fim do expediente comercial), antes do pico das 19h, quando a carga residencial se soma ao fim do dia de trabalho. Sábado e domingo/feriado têm o pico na mesma hora (19h), mas em nível menor (sábado 46,7 mil e domingo/feriado 44,2 mil, contra 49,2 mil no dia útil: −5% e −10%), e manhã e tarde bem mais baixas (às 11h, −11% e −21% contra o dia útil), porque a atividade comercial e industrial cai.

A meia-noite de sábado (40,0 mil) fica acima da média do dia útil (39,2 mil) por causa da segunda-feira: a madrugada de segunda, que vem de um domingo calmo, puxa a média para baixo (36,9 mil), enquanto de terça a sexta a meia-noite fica em 39,6–40,0 mil, igual à do sábado.

Como os dados vão de 2021 a 2025, a MMGD (solar) entra na carga a partir de abr/2023 e tende a inflar as horas de sol: comparando dias úteis de 2024–25 com 2021–22, a carga subiu 10–11% à noite e no fim do dia (17h–23h) e 15–17% de 8h às 13h, com pico às 11h–12h, embora isso se misture com o crescimento geral.

Para o projeto: o pico de consumo coincide com o fim da tarde, então o perfil horário do consumidor importa para ponderar o PLD, e a escolha de usar a carga média diária na curva do supermercado evita herdar essa distorção da MMGD dentro do dia.

## 3. Carga contra temperatura (SE/CO), por estação do ano

**Fonte:** `fct_submercado_horario` (carga do ONS e temperatura do INMET na mesma hora) com
`dim_tempo`. **Método:** cada ponto é **um dia**: média diária da carga (MWmed) e da temperatura
(°C, média simples dos estados do SE/CO, provisória até os pesos da EPE), só dias com pelo menos
20 horas com carga e temperatura. Um painel por estação do ano (hemisfério sul; verão = dez a fev),
com os mesmos eixos; os pontos cinza ao fundo são os dias das outras estações. Com
`SOMENTE_DIAS_UTEIS = True` (padrão) só entram dias úteis, porque o fim de semana e os feriados
têm carga bem menor (gráfico 2). **Nota:** a carga muda de definição em 29/04/2023 (MMGD) dentro do
período; o gráfico não corrige isso.

In [ ]:
SOMENTE_DIAS_UTEIS = True

filtro_dia = "AND t.tipo_dia = 'dia_util'" if SOMENTE_DIAS_UTEIS else ""
dispersao = sessao.consultar(
    "3. carga x temperatura (média diária)",
    f"""
    SELECT t.data_local,
           ANY_VALUE(t.estacao_do_ano) AS estacao,
           AVG(f.carga_mwmed) AS carga_media,
           AVG(f.temperatura_c) AS temperatura_media,
           COUNT(*) AS horas
    FROM {T("fct_submercado_horario")} AS f
    JOIN {T("dim_tempo")} AS t USING (instante_utc)
    WHERE f.codigo_submercado = 'SE'
      AND f.carga_mwmed IS NOT NULL
      AND f.temperatura_c IS NOT NULL
      {filtro_dia}
    GROUP BY t.data_local
    HAVING COUNT(*) >= 20
    ORDER BY t.data_local
    """,
)
for coluna in ("carga_media", "temperatura_media"):
    dispersao[coluna] = dispersao[coluna].astype(float)

ESTACOES = [
    ("verao", "Verão"),
    ("outono", "Outono"),
    ("inverno", "Inverno"),
    ("primavera", "Primavera"),
]
fig, eixos = plt.subplots(2, 2, figsize=(9.5, 7.2), sharex=True, sharey=True)
for ax, (chave, rotulo) in zip(eixos.flat, ESTACOES, strict=True):
    ax.grid(axis="x", color=u.GRADE, linewidth=0.8)
    ax.scatter(
        dispersao["temperatura_media"], dispersao["carga_media"], s=5, color="#cfcec9", linewidths=0
    )
    d = dispersao[dispersao["estacao"] == chave]
    ax.scatter(
        d["temperatura_media"], d["carga_media"], s=9, color=u.AZUL, alpha=0.65, linewidths=0
    )
    ax.set_title(f"{rotulo} ({len(d)} dias)", fontsize=10.5)
for ax in eixos[1]:
    ax.set_xlabel("temperatura média do dia (°C)")
for ax in eixos[:, 0]:
    ax.set_ylabel("carga média do dia (MWmed)")
tipo = "dias úteis" if SOMENTE_DIAS_UTEIS else "todos os dias"
fig.suptitle(
    f"Carga do SE/CO contra a temperatura, {tipo}",
    x=0.01,
    ha="left",
    fontsize=12,
    fontweight="bold",
)
fig.tight_layout()
u.salvar_figura(fig, "03_carga_x_temperatura")
plt.show()

### Minhas conclusões

Existe uma relação forte e positiva entre temperatura e carga nos dias úteis. O número principal é a inclinação **controlando o ano: ~1,1 mil MWmed por °C (~2,5% por °C)**, ajustada sobre todos os dias. Controlar o ano muda pouco a inclinação (1,17 mil sem controle): o ano desloca o nível da carga (a média do recorte vai de 40,6 mil em 2021 a 46,1 mil em 2024), mas pouco a sensibilidade à temperatura.

Entre 20 °C e 26 °C, onde está a maior parte dos dias (81%), a inclinação é maior (~1,3 mil MWmed/°C, 3,0% por °C, controlando o ano) e a carga média vai de ~40,7 mil a 20 °C para ~49,0 mil a 26 °C. A relação fica mais inclinada nas temperaturas altas (sem controlar o ano: ~0,4 mil MWmed/°C entre 16 e 20 °C, ~0,8 mil entre 20 e 23 °C e ~1,75 mil entre 23 e 26 °C), o que é coerente com o acionamento de ar-condicionado e refrigeração, e volta a ~0,9 mil acima de 26 °C, onde há poucos dias. O inverno concentra os dias frios e de carga baixa (média de 20,2 °C e 41,0 mil MWmed); o verão, os dias quentes e de carga alta (24,2 °C e 45,4 mil).

Ressalva: os pontos misturam 2021 a 2026, período em que a carga cresceu e mudou de definição (MMGD), então parte da dispersão vertical é efeito do ano, e não da temperatura.

Para o projeto: a temperatura não entra na previsão de 12 meses (não se conhece o clima do ano seguinte), mas é a principal candidata para explicar os erros mensais, como ondas de calor e invernos atípicos.

## 4. Distribuição do PLD horário do SUDESTE por ano, 2021 a 2026

**Fonte:** `fct_pld_horario` (submercado `SE`) com `dim_tempo` para o ano local.
**Método:** à esquerda, a distribuição das horas de cada ano (caixa = quartis, traço = mediana,
hastes = percentis 1 e 99; escala logarítmica) e o **piso do ano** (losango). À direita, a
porcentagem de horas do ano **no piso**. O piso aqui é o **menor PLD do ano**, que coincide com o
piso regulatório (a confirmar na ANEEL). 2026 vai até o fim dos dados e é parcial.

In [ ]:
pld = sessao.consultar(
    "4. PLD horário do SE",
    f"""
    SELECT t.ano, p.pld_rs_mwh
    FROM {T("fct_pld_horario")} AS p
    JOIN {T("dim_tempo")} AS t USING (instante_utc)
    WHERE p.codigo_submercado = 'SE'
    """,
)
pld["pld_rs_mwh"] = pld["pld_rs_mwh"].astype(float)
anos = sorted(pld["ano"].unique())
por_ano = {a: pld.loc[pld["ano"] == a, "pld_rs_mwh"].to_numpy() for a in anos}
piso = {a: v.min() for a, v in por_ano.items()}
no_piso = {a: float(np.mean(v <= piso[a] + 0.005) * 100) for a, v in por_ano.items()}
rotulos = [f"{a}*" if a == anos[-1] else str(a) for a in anos]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [3, 2]})
caixas = ax1.boxplot(
    [por_ano[a] for a in anos],
    whis=(1, 99),
    showfliers=False,
    patch_artist=True,
    widths=0.55,
    medianprops={"color": u.TINTA, "linewidth": 1.8},
    whiskerprops={"color": u.TINTA_3},
    capprops={"color": u.TINTA_3},
    boxprops={"edgecolor": u.AZUL, "facecolor": "#cfe0f5"},
)
ax1.plot(
    range(1, len(anos) + 1),
    [piso[a] for a in anos],
    "D",
    color=u.TINTA_2,
    ms=6,
    mfc=u.SUPERFICIE,
    mew=1.5,
    label="piso do ano (menor PLD)",
)
ax1.set_yscale("log")
ax1.set_xticks(range(1, len(anos) + 1), rotulos)
ax1.set_ylabel("PLD horário (R$/MWh, escala log)")
ax1.set_title("Distribuição do PLD do SUDESTE por ano")
ax1.legend(loc="upper right", fontsize=9)
ax2.barh(rotulos, [no_piso[a] for a in anos], color=u.AZUL, height=0.6)
ax2.invert_yaxis()
ax2.grid(axis="y", visible=False)
ax2.grid(axis="x", color=u.GRADE)
for y, a in enumerate(anos):
    ax2.annotate(
        f"{no_piso[a]:.1f}%",
        (no_piso[a], y),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=u.TINTA_2,
    )
ax2.set_xlim(0, 110)
ax2.set_xlabel("% das horas do ano no piso")
ax2.set_title("Tempo no piso")
fig.text(
    0.01,
    -0.02,
    "* ano parcial. Fonte: CCEE (PLD horário), via marts.fct_pld_horario.",
    fontsize=8,
    color=u.TINTA_3,
)
fig.tight_layout()
u.salvar_figura(fig, "04_pld_se_por_ano")
plt.show()
pd.DataFrame(
    {
        "horas": {a: len(v) for a, v in por_ano.items()},
        "piso (R$/MWh)": piso,
        "% no piso": {a: round(p, 1) for a, p in no_piso.items()},
    }
)

### Minhas conclusões

O PLD do SUDESTE mudou de regime de um ano para o outro. Em 2021 (crise hídrica) a mediana ficou em R$ 202/MWh e 27% das horas passaram de R$ 500. Em 2022 e 2023, com reservatórios cheios, o preço ficou no piso em 79% e 98% das horas. Em 2024 voltaram os picos fortes (até R$ 1.471/MWh), mesmo com 64% das horas no piso. Em 2025 a mediana foi de R$ 269 (65% das horas acima de R$ 200, mas só 0,3% acima de R$ 500) e em 2026, até agora, de R$ 193 (48% das horas acima de R$ 200); nos dois anos cerca de 23% das horas ficaram no piso.

O piso muda a cada ano (R$ 49,77 em 2021, R$ 55,70 em 2022, R$ 69,04 em 2023, R$ 61,07 em 2024, R$ 58,60 em 2025 e R$ 57,31 em 2026), o que é coerente com um valor regulatório definido anualmente pela ANEEL (os valores exatos ainda precisam ser confirmados).

Para o projeto: o resultado do backtest vai depender muito do regime de cada ano. Em anos de piso, contratar acima do necessário custa caro, porque a sobra é liquidada a um preço muito baixo; em anos de crise, ficar descoberto custa caro. Por isso os resultados precisam ser lidos ano a ano, e não só pela média.

## 5. PLD médio anual dos quatro submercados, 2001 a 2026

**Fonte:** `fct_pld_semanal` (2001 a 2020) e `fct_pld_horario` (2021 em diante), os dois com o
código do submercado do ONS. **Método:** até 2020, a média do preço de cada semana (média simples
dos 3 patamares, que o arquivo não identifica) **ponderada pelas horas da semana que caem em cada
ano local**; de 2021 em diante, a média das horas do ano. Linha tracejada = série semanal; linha
cheia = série horária (não há linha unindo 2020 a 2021, porque a fonte muda). O SUDESTE está em
destaque. **2001** só tem dados de 30/06 em diante e **2026** vai até o fim dos dados (marcador vazado).

In [ ]:
pld_anual = sessao.consultar(
    "5. PLD médio anual",
    f"""
    WITH semana AS (
        SELECT codigo_submercado, inicio_semana_utc, fim_semana_utc, AVG(pld_rs_mwh) AS preco
        FROM {T("fct_pld_semanal")}
        GROUP BY codigo_submercado, inicio_semana_utc, fim_semana_utc
    ),
    ano_local AS (
        SELECT ano,
               TIMESTAMP(DATE(ano, 1, 1), 'America/Sao_Paulo') AS ini,
               TIMESTAMP(DATE(ano + 1, 1, 1), 'America/Sao_Paulo') AS fim
        FROM UNNEST(GENERATE_ARRAY(2001, 2020)) AS ano
    ),
    semanal AS (
        SELECT s.codigo_submercado, a.ano,
               SAFE_DIVIDE(SUM(s.preco * horas), SUM(horas)) AS pld_medio,
               SUM(horas) AS horas,
               'semanal' AS fonte
        FROM semana AS s
        JOIN ano_local AS a ON s.inicio_semana_utc < a.fim AND s.fim_semana_utc > a.ini
        CROSS JOIN UNNEST([TIMESTAMP_DIFF(
            LEAST(s.fim_semana_utc, a.fim), GREATEST(s.inicio_semana_utc, a.ini), HOUR
        )]) AS horas
        GROUP BY s.codigo_submercado, a.ano
    ),
    horario AS (
        SELECT p.codigo_submercado, t.ano, AVG(p.pld_rs_mwh) AS pld_medio,
               COUNT(*) AS horas, 'horario' AS fonte
        FROM {T("fct_pld_horario")} AS p
        JOIN {T("dim_tempo")} AS t USING (instante_utc)
        GROUP BY p.codigo_submercado, t.ano
    )
    SELECT * FROM semanal
    UNION ALL
    SELECT * FROM horario
    ORDER BY codigo_submercado, ano
    """,
)
pld_anual["pld_medio"] = pld_anual["pld_medio"].astype(float)
ROTULO_SUBMERCADO = {"SE": "SE/CO", "S": "S", "NE": "NE", "N": "N"}
parcial = {2001, int(pld_anual["ano"].max())}

fig, ax = plt.subplots(figsize=(11, 4.8))
finais = []
for cod in ["N", "NE", "S", "SE"]:  # o SE por último para ficar por cima
    destaque = cod == "SE"
    cor, largura = (u.AZUL, 2.4) if destaque else (u.TINTA_3, 1.1)
    d = pld_anual[pld_anual["codigo_submercado"] == cod]
    for fonte, estilo in (("semanal", (0, (4, 2.5))), ("horario", "-")):
        seg = d[d["fonte"] == fonte]
        ax.plot(
            seg["ano"],
            seg["pld_medio"],
            color=cor,
            linewidth=largura,
            linestyle=estilo,
            alpha=1 if destaque else 0.8,
        )
    for ano in parcial:
        p = d[d["ano"] == ano]
        ax.plot(p["ano"], p["pld_medio"], "o", mfc=u.SUPERFICIE, mec=cor, mew=1.5, ms=6, zorder=5)
    finais.append((d["pld_medio"].iloc[-1], cod, destaque))
afastamento = 0.05 * pld_anual["pld_medio"].max()  # afasta rótulos que se encostariam
posicoes, anterior = [], -1e9
for valor, cod, destaque in sorted(finais):
    y = max(valor, anterior + afastamento)
    posicoes.append((y, cod, destaque))
    anterior = y
for y, cod, destaque in posicoes:
    ax.annotate(
        ROTULO_SUBMERCADO[cod],
        (pld_anual["ano"].max(), y),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=9.5,
        va="center",
        color=u.AZUL if destaque else u.TINTA_2,
        fontweight="bold" if destaque else "normal",
    )
ax.plot([], [], color=u.TINTA_2, linestyle=(0, (4, 2.5)), label="semanal (ponderada por ano)")
ax.plot([], [], color=u.TINTA_2, label="horária")
ax.plot([], [], "o", mfc=u.SUPERFICIE, mec=u.TINTA_2, label="ano parcial")
ax.set_xlim(2000.5, 2027.3)
ax.set_xticks(range(2001, 2027, 2))
ax.set_ylabel("PLD médio do ano (R$/MWh)")
ax.set_title("PLD médio anual por submercado, 2001 a 2026 (SE/CO em destaque)")
ax.legend(loc="upper right", fontsize=9)
fig.text(
    0.01,
    -0.02,
    "Fonte: CCEE (PLD semanal 2001-2020 e horário 2021+), via marts (fct_pld_*).",
    fontsize=8,
    color=u.TINTA_3,
)
u.salvar_figura(fig, "05_pld_medio_anual")
plt.show()

### Minhas conclusões

Em 25 anos, o PLD médio anual do SE/CO teve picos fortes em 2001 (racionamento, ~R$ 494, considerando só a parte do ano coberta pela série, de 30/06 em diante) e em 2014 (~R$ 690), intercalados com vários anos de preço baixo. Outros anos ficam em patamar bem menor: 2013 e 2015 (R$ 264 e R$ 288), 2017 e 2018 (R$ 325 e R$ 289) e 2021 (R$ 280). Os picos acompanham a hidrologia: anos de pouca chuva e reservatórios baixos.

Em 2001 o Sul ficou com preço baixo (R$ 43) enquanto SE/CO e Norte (R$ 494) e Nordeste (R$ 617) explodiam, o que mostra que os submercados podem se descolar quando há limite de transmissão entre eles. A quebra da linha entre 2020 e 2021 marca a mudança da série semanal para a horária.

Para o projeto: os cenários de PLD da otimização precisam do histórico longo, com 2001 e 2014 incluídos. Usando só 2021 a 2025, os cenários subestimariam os preços extremos, o CVaR ficaria baixo demais e a recomendação de contrato ficaria exposta justamente nos anos que mais custam caro.

## Custo das consultas

In [ ]:
sessao.resumo_de_custo()